In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Load dataset
df = pd.read_csv("Language Detection.csv")
X = df['Text'].astype(str)
y = df['Language']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# =================== 1. Naive Bayes + N-Gram ========================
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, norm='l2')),
    ('nb', MultinomialNB())
])
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average='weighted')
prec_nb = precision_score(y_test, y_pred_nb, average='weighted')

# =================== 2. CNN + N-Gram ================================
tokenizer_ngram = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer_ngram.fit_on_texts(X_train)
X_train_seq_ngram = tokenizer_ngram.texts_to_sequences(X_train)
X_test_seq_ngram = tokenizer_ngram.texts_to_sequences(X_test)
X_train_pad_ngram = pad_sequences(X_train_seq_ngram, maxlen=100, padding='post')
X_test_pad_ngram = pad_sequences(X_test_seq_ngram, maxlen=100, padding='post')

y_train_cat_ngram = to_categorical(y_train_enc)
y_test_cat_ngram = to_categorical(y_test_enc)

cnn_ngram_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])
cnn_ngram_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn_ngram_model.fit(X_train_pad_ngram, y_train_cat_ngram, epochs=5, batch_size=32, validation_split=0.1)

y_pred_cnn_ngram = cnn_ngram_model.predict(X_test_pad_ngram)
y_pred_cnn_ngram_labels = le.inverse_transform(np.argmax(y_pred_cnn_ngram, axis=1))

acc_cnn_ngram = accuracy_score(y_test, y_pred_cnn_ngram_labels)
f1_cnn_ngram = f1_score(y_test, y_pred_cnn_ngram_labels, average='weighted')
prec_cnn_ngram = precision_score(y_test, y_pred_cnn_ngram_labels, average='weighted')

# =================== 3. Naive Bayes + Feature Extraction ================
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb_feat_model = MultinomialNB()
nb_feat_model.fit(X_train_tfidf, y_train_enc)
y_pred_nb_feat = nb_feat_model.predict(X_test_tfidf)

acc_nb_feat = accuracy_score(y_test_enc, y_pred_nb_feat)
f1_nb_feat = f1_score(y_test_enc, y_pred_nb_feat, average='weighted')
prec_nb_feat = precision_score(y_test_enc, y_pred_nb_feat, average='weighted')

# =================== 4. CNN + Feature Extraction =======================
tokenizer_feat = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer_feat.fit_on_texts(X)
sequences_feat = tokenizer_feat.texts_to_sequences(X)
padded_feat = pad_sequences(sequences_feat, maxlen=100)
labels_feat = le.transform(y)

X_train_feat, X_test_feat, y_train_feat, y_test_feat = train_test_split(padded_feat, labels_feat, test_size=0.2, random_state=42)

cnn_feat_model = Sequential()
cnn_feat_model.add(Embedding(input_dim=10000, output_dim=128, input_length=100))
cnn_feat_model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
cnn_feat_model.add(GlobalMaxPooling1D())
cnn_feat_model.add(Dropout(0.3))
cnn_feat_model.add(Dense(64, activation='relu'))
cnn_feat_model.add(Dense(len(le.classes_), activation='softmax'))
cnn_feat_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn_feat_model.fit(X_train_feat, y_train_feat, epochs=5, batch_size=32, validation_split=0.1)

y_pred_probs_feat = cnn_feat_model.predict(X_test_feat)
y_pred_feat = np.argmax(y_pred_probs_feat, axis=1)

acc_cnn_feat = accuracy_score(y_test_feat, y_pred_feat)
f1_cnn_feat = f1_score(y_test_feat, y_pred_feat, average='weighted')
prec_cnn_feat = precision_score(y_test_feat, y_pred_feat, average='weighted')

# =================== Results Summary ===========================
results = [
    {"Model": "Naive Bayes + N-Gram", "Accuracy": acc_nb, "F1 Score": f1_nb, "Precision": prec_nb},
    {"Model": "CNN + N-Gram", "Accuracy": acc_cnn_ngram, "F1 Score": f1_cnn_ngram, "Precision": prec_cnn_ngram},
    {"Model": "Naive Bayes + Feature Extraction", "Accuracy": acc_nb_feat, "F1 Score": f1_nb_feat, "Precision": prec_nb_feat},
    {"Model": "CNN + Feature Extraction", "Accuracy": acc_cnn_feat, "F1 Score": f1_cnn_feat, "Precision": prec_cnn_feat}
]

comparison_df = pd.DataFrame(results)
print("\nModel Performance Comparison:")
print(comparison_df)

# =================== Prediction Function ===========================
def predict_language(input_text):
    print(f"\nInput Sentence: {input_text}\n")

    # 1. Naive Bayes + N-Gram
    nb_ngram_pred = nb_pipeline.predict([input_text])[0]
    print(f"Naive Bayes + N-Gram: {nb_ngram_pred}")

    # 2. CNN + N-Gram
    input_seq_ngram = tokenizer_ngram.texts_to_sequences([input_text])
    input_pad_ngram = pad_sequences(input_seq_ngram, maxlen=100, padding='post')
    cnn_ngram_probs = cnn_ngram_model.predict(input_pad_ngram)
    cnn_ngram_pred = le.inverse_transform([np.argmax(cnn_ngram_probs)])[0]
    print(f"CNN + N-Gram: {cnn_ngram_pred}")

    # 3. Naive Bayes + Feature Extraction
    input_tfidf_feat = tfidf_vectorizer.transform([input_text])
    nb_feat_pred = le.inverse_transform(nb_feat_model.predict(input_tfidf_feat))[0]
    print(f"Naive Bayes + Feature Extraction: {nb_feat_pred}")

    # 4. CNN + Feature Extraction
    input_seq_feat = tokenizer_feat.texts_to_sequences([input_text])
    input_pad_feat = pad_sequences(input_seq_feat, maxlen=100)
    cnn_feat_probs = cnn_feat_model.predict(input_pad_feat)
    cnn_feat_pred_index = np.argmax(cnn_feat_probs)
    cnn_feat_pred = le.inverse_transform([cnn_feat_pred_index])[0]
    print(f"CNN + Feature Extraction: {cnn_feat_pred}")

# =================== Example Demo ===========================
predict_language("Bonjour, comment allez-vous aujourd'hui?")

Epoch 1/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.3021 - loss: 2.2355 - val_accuracy: 0.9359 - val_loss: 0.2722
Epoch 2/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 10s 43ms/step - accuracy: 0.9520 - loss: 0.2015 - val_accuracy: 0.9492 - val_loss: 0.1456
Epoch 3/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.9770 - loss: 0.0853 - val_accuracy: 0.9661 - val_loss: 0.1237
Epoch 4/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.9772 - loss: 0.0735 - val_accuracy: 0.9589 - val_loss: 0.1341
Epoch 5/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.9800 - loss: 0.0669 - val_accuracy: 0.9661 - val_loss: 0.1125
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Epoch 1/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - accuracy: 0.2685 - loss: 2.3416 - val_accuracy: 0.9238 - val_loss: 0.4155
Epoch 2/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.9230 - loss: 0.3403 - val_accuracy: 0.9601 - val_loss: 0.1454
Epoch 3/5
233/233 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - ac